In [1]:
from pathlib import Path
from pydantic import BaseModel, Field
from typing import List
from llama_cloud.client import LlamaCloud
from llama_index.core.prompts import PromptTemplate
from llama_index.core.async_utils import run_jobs
import os
import json
from dotenv import load_dotenv
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
import chromadb
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore
# from llama_index.embeddings.huggingface_api import HuggingFaceInferenceAPIEmbedding
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_cloud_services import LlamaExtract
from typing import List, Dict




llama_extract = LlamaExtract(api_key=os.getenv("LAMMA_CLOUD_API_KEY"))
load_dotenv(dotenv_path=r"C:\Users\ogida\Desktop\Hope work\Tech\Vscode_files\ScaleSense\.env")


c:\Users\ogida\Desktop\Hope work\Tech\Vscode_files\ScaleSense\scale\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")



# global_skills = []
# global_countries = []
# global_domains = []
# global_years_of_experience = []

# 1. Domains
global_domains = [
    "Information Technology", "Finance", "Financial Trading", "Accounting", 
    "Human Resources", "Engineering", "Robotics", "Healthcare", "Sales", 
    "Marketing", "Operations", "Product Management", "Legal"
]

# 2. Countries
global_countries = [
    "United States", "United Kingdom", "Canada", "Australia", "India", 
    "Germany", "France", "Singapore", "Nigeria", "Remote"
]

# 3. Skills
global_skills = [
    # Software & Cloud
    "Python", "Java", "C++", "C#", "JavaScript", "TypeScript", "Go", "Rust", 
    "React", "Angular", "Vue.js", "Node.js", "Django", "Flask", "Framer",
    "AWS", "Azure", "Google Cloud (GCP)", "Docker", "Kubernetes", "CI/CD", "Linux",
    
    # Data & AI
    "SQL", "NoSQL", "Machine Learning", "MLOps", "MLflow", "PyTorch", "TensorFlow", 
    "Data Engineering", "Pandas", "Computer Vision", "Large Language Models (LLMs)", "RAG",
    
    # Finance & Business
    "Financial Modeling", "Risk Management", "Forex Trading", "Accounting", 
    "SOX Compliance", "Excel", "Data Analysis", "Project Management", "Agile"
]

# 4. Years of Experience (Keep this as a string!)
# Experience is a continuous number, so it is much better to give the LLM an 
# instruction rather than a list like [1, 2, 3, 4, 5]. 
global_years_of_experience = (
    "Extract as a single integer representing the minimum years requested. "
    "For example: '5' for '5+ years', '0' for 'entry level'. Do not include text."
)

In [3]:
class Metadata(BaseModel):
    """
    A data model representing key professional and educational metadata extracted from a resume.
    This class captures essential candidate information including technical/professional skills, years of experience 
    based on the work experience in the resume, the professional domain they belong to,
    and the geographical distribution of their educational background.

    Attributes:
        skills (List[str]): Technical and professional competencies of the candidate
        country (List[str]): Countries where the candidate pursued formal education
        Years of Experience (int): Total years of professional experience in the relevant domain
        domain (str): The professional domain of the candidate (e.g., SALES, IT,

    Example:
        {
            "skills": ["Python", "Machine Learning", "SQL", "Project Management"],
            "country": ["United States", "India"],
            "domain": "Information Technology",
            "Years of Experience": 5,
        }
    """

    domain: str = Field(...,
                        description="The domain of the candidate can be one of SALES/ IT/ FINANCE"
                                    "Returns an empty string if no domain is identified.")

    skills: List[str] = Field(
        ...,
        description="List of technical, professional, and soft skills extracted from the resume. "
                   "and domain expertise. Returns an empty list if no skills are identified."
    )

    country: List[str] = Field(
        ...,
        description="List of countries where the candidate completed their formal education, Only extract the country."
                   "Returns an empty list if countries are not specified."
    )
    years_of_experience: int = Field(
        ...,
        description="Total years of professional experience in the relevant domain, calculated based on the work experience section of the resume. "
                   "Returns 0 if no work experience is identified."
    )







async def get_metadata(text):
    """Function to get the metadata from the given resume of the candidate"""
    prompt_template = PromptTemplate("""Generate skills, and country of the education for the given candidate resume.

    Resume of the candidate:

    {text}""")

    metadata = await llm.astructured_predict(
        Metadata,
        prompt_template,
        text=text,
    )

    return metadata

In [4]:
# await upload_documents(client, pipeline, documents)

## Connect to llamma chroma db and make queries

In [6]:
# 1. Initialize your ChromaDB Client and Vector Store
db = chromadb.PersistentClient(path="../chroma_db")
chroma_collection = db.get_or_create_collection("parsed_documents")


# 2. Define the Vector Store (Renamed from 'index' to 'vector_store')
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# 3. Load the EXACT same embedding model you used to ingest the resumes
local_embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# 4. Build the actual Index
# This is what gives you access to the .as_retriever() method!
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    embed_model=local_embed_model
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 347.26it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
from llama_index.core.vector_stores import (
    MetadataFilter,
    MetadataFilters,
    FilterOperator,
    FilterCondition
)
async def get_query_metadata(text):
    """Function to get the metadata from the given user query"""
    prompt_template = PromptTemplate("""Generate skills, and country of the education for the given user query.

    Extracted metadata should be from the following items:

    skills: {global_skills}
    countries: {global_countries}
    domains: {global_domains}
    years of experience: {global_years_of_experience}
    user query:

    {text}""")

    extracted_metadata = await llm.astructured_predict(
        Metadata,
        prompt_template,
        text=text,
        global_skills=global_skills,
        global_countries=global_countries,
        global_domains=global_domains,
        global_years_of_experience=global_years_of_experience
    )

    return extracted_metadata




def get_candidates_file_paths(candidates):

  file_paths = []
  for candidate in candidates:
    file_paths.append(candidate['file_path'])

  return list(set(file_paths))


def get_jd_candidates_file_paths(candidates):

  file_paths = []
  for candidate in candidates:
        node = candidate.node
        file_path = node.metadata.get('file_path')
        if file_path:       
            file_paths.append(file_path)

    
  return list(set(file_paths))



async def candidates_retriever_from_jd(job_description: str):
    # Use structured predict to infer the metadata filters and query string.
    metadata_info = await get_metadata(job_description)
    filters = MetadataFilters(
    filters=[
        MetadataFilter(key="domain", operator=FilterOperator.EQ, value=metadata_info.domain),
        MetadataFilter(key="country", operator=FilterOperator.IN, value=metadata_info.country),
        MetadataFilter(key="skills", operator=FilterOperator.IN, value=metadata_info.skills),
        MetadataFilter(key="years_of_experience", operator=FilterOperator.GTE, value=metadata_info.years_of_experience)
    ],
    condition=FilterCondition.OR
)
    print(f"> Inferred filters: {filters.json()}")
    retriever = index.as_retriever(
    retrieval_mode="chunks",
    metadata_filters=filters,
    )
    # run query
    return retriever.retrieve(job_description)

In [8]:
def deduplicate_candidates_by_filepath(retrieved_nodes):
    """
    Takes a list of LlamaIndex NodeWithScore objects and deduplicates them 
    based on the 'file_path' metadata, keeping the highest scoring chunk.
    """
    unique_candidates = {}
    for node_with_score in retrieved_nodes:
        node = node_with_score.node
        file_path = node.metadata.get('file_path')
        # Failsafe: Skip if the chunk somehow doesn't have a file path
        if not file_path:
            continue

        # Because nodes are already sorted by score, the first time we see 
        # a file_path, it is guaranteed to be the most relevant chunk.
        if file_path not in unique_candidates:
            unique_candidates[file_path] = {
                "file_path": file_path,
                "domain": node.metadata.get('domain', 'N/A'),
                "years_of_experience": node.metadata.get('years_of_experience', 0),
                "skills": node.metadata.get('skills', ''),
                "match_score": round(node_with_score.score, 4), 
                # "relevant_excerpt": node.text[:400].strip() + "..."
                "relevant_excerpt": node.text
            }

    # Return the clean dictionary values as a standard Python list
    return list(unique_candidates.values())




async def candidates_retriever_from_query(query: str):
    """Synthesizes an answer to your question by feeding in an entire relevant document as context."""
    print(f"> User query string: {query}")
    # Use structured predict to infer the metadata filters and query string.
    metadata_info = await get_query_metadata(query)

    # 1. Start with an empty list of filters
    active_filters = []

    # 2. Dynamically check what the LLM extracted (metadata_info) and add only valid filters
    if metadata_info.domain:
        active_filters.append(
            MetadataFilter(key="domain", operator=FilterOperator.EQ, value=metadata_info.domain)
        )

    if metadata_info.skills and len(metadata_info.skills) > 0:
        active_filters.append(
            MetadataFilter(key="skills", operator=FilterOperator.IN, value=metadata_info.skills)
        )

    if metadata_info.years_of_experience and metadata_info.years_of_experience > 0:
        active_filters.append(
            MetadataFilter(key="years_of_experience", operator=FilterOperator.GTE, value=metadata_info.years_of_experience)
        )

    if metadata_info.country and len(metadata_info.country) > 0:
        active_filters.append(
            MetadataFilter(key="country", operator=FilterOperator.IN, value=metadata_info.country)
        )

    # 3. Assemble the final MetadataFilters object strictly using AND
    # If the user only asked for "Finance" and "2 years experience", 
    # active_filters will only contain those two rules.
    filters = MetadataFilters(
        filters=active_filters,
        condition=FilterCondition.OR
    )

    retriever = index.as_retriever(
    retrieval_mode="chunks",
    metadata_filters=filters,
    )
    # run query
    result = retriever.retrieve(query)

    # 2. Pass them through your new deduplication filter
    clean_candidates = deduplicate_candidates_by_filepath(result)

    print(f"> Inferred filters: {filters.model_dump_json()}")
    return clean_candidates

In [9]:
# query = "I want a IT person with  python skill and at least two years of experience "

query = "I want  people in technology "
nodes = await candidates_retriever_from_query(query)

> User query string: I want  people in technology 
> Inferred filters: {"filters":[{"key":"domain","value":"Information Technology","operator":"=="}],"condition":"or"}


In [10]:
nodes

[{'file_path': 'C:\\Users\\ogida\\Desktop\\Hope work\\Tech\\Vscode_files\\ScaleSense\\src\\data\\sampled_data\\IT\\52246737.pdf',
  'domain': 'Information Technology, Information Security',
  'years_of_experience': 5,
  'skills': 'A Certification, Active Directory, AD, Anti-Virus, BUSINESS PROCESS, coaching, Compliance Manager, CA, hardware, Concept, CONTRACT NEGOTIATIONS, contracts, conversion, encryption, clients, designing, desktops, Disaster Recovery, DNS, Firewall, functional, Gateway, IDS, imaging, Information Security, laptops, Legal, Linux, Mac, director, McAfee, mediator, mentoring, Exchange, Microsoft Exchange, Mail, Windows (XP), negotiation, Enterprise, network, Networking, operating systems, personnel, policies, processes, programming, Proxy, Red Hat, RELATIONSHIP BUILDING, Risk Management, SAN, statistics, Symantec, TCP/IP, Technical Trainer, phones, troubleshooting, VPN, Author',
  'match_score': 0.3062,
  'relevant_excerpt': '"SAN",\n      "statistics",\n      "Symantec

In [11]:
print(get_candidates_file_paths(nodes))

['C:\\Users\\ogida\\Desktop\\Hope work\\Tech\\Vscode_files\\ScaleSense\\src\\data\\sampled_data\\IT\\52246737.pdf']


## Extracting JD

In [12]:

from llama_index.readers.file import PyMuPDFReader


# agent = llama_extract.get_agent("resume-screening")


file_path = r"C:\Users\ogida\Desktop\Hope work\Tech\Vscode_files\ScaleSense\src\data\job_description.pdf"


reader = PyMuPDFReader()

# Extract the data
documents = reader.load_data(file_path=file_path)

# Combine all pages into one single string if you want to pass it to your LLM
full_jd_text = "\n".join([doc.text for doc in documents])


print(full_jd_text[:500])




Senior Information Technology Manager
 
About the Role
We are seeking an experienced Information Technology Manager to lead our technology initiatives and drive digital transformation across the 
organization. The ideal candidate will combine strong technical expertise with business acumen and leadership skills.
 
Key Responsibilities
- Lead and manage a cross-functional IT team in developing and implementing technology solutions
- Oversee the planning, implementation, and maintenance of enterpr


In [13]:

job_description = full_jd_text

candidates_based_on_jd = await candidates_retriever_from_jd(job_description)

2026-03-05 13:48:30,409 - INFO - HTTP Request: POST https://router.huggingface.co/v1/chat/completions "HTTP/1.1 200 OK"


> Inferred filters: {"filters":[{"key":"domain","value":"Information Technology","operator":"=="},{"key":"country","value":[],"operator":"in"},{"key":"skills","value":["Enterprise Resource Planning (ERP) systems","Network infrastructure and security","Cloud computing platforms and services","IT service management frameworks","Database management systems","System integration and architecture","Virtualization technologies","Disaster recovery and business continuity","Team management and development","Strategic planning and execution","Strong communication and presentation skills","Problem-solving and analytical thinking","Change management","Budget management","Stakeholder management","Cross-functional collaboration","Project management and delivery","IT security, compliance, and risk management","Digital transformation initiatives","Agile methodologies"],"operator":"in"},{"key":"years_of_experience","value":8,"operator":">="}],"condition":"or"}


C:\Users\ogida\AppData\Local\Temp\ipykernel_19860\2551611132.py:71: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  print(f"> Inferred filters: {filters.json()}")


In [14]:
candidates_based_on_jd

[NodeWithScore(node=TextNode(id_='4a5b6580-cd89-4d7f-971e-46599781c23e', embedding=None, metadata={'skills': 'Strategic Planning, Project and Program Management, Change Implementation, Team Leadership, Customer Orientation, Time and Resource Optimization, Business Processes, Management, Budgeting, Business Office Operations, Data Processing, Hardware Platforms, Enterprise Software Applications, Outsourced Systems, Cloud SaaS and IaaS, Systems Design and Development, IT Planning and Development, Project Management Principles, Human Resource Management Principles', 'country': None, 'domain': None, 'years_of_experience': 40, 'file_path': 'C:\\Users\\ogida\\Desktop\\Hope work\\Tech\\Vscode_files\\ScaleSense\\src\\data\\sampled_data\\IT\\18159866.pdf'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=['file_path'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='81053d45-7d0b-48af-a146-18c1b467f284', node_type='4', metadata={'skills': 'Strategic Planning,

In [15]:
candidates_file_paths = get_jd_candidates_file_paths(candidates_based_on_jd)
candidates_file_paths

['C:\\Users\\ogida\\Desktop\\Hope work\\Tech\\Vscode_files\\ScaleSense\\src\\data\\sampled_data\\IT\\38753827.pdf',
 'C:\\Users\\ogida\\Desktop\\Hope work\\Tech\\Vscode_files\\ScaleSense\\src\\data\\sampled_data\\IT\\18159866.pdf']

## Analyse candidate resume based on retrieval

In [16]:


#Function to extract the text from candidates resume

def batch_extract_pdfs(file_paths: List[str]) -> Dict[str, str]:
    """
    Extracts text from a list of PDF file paths using LlamaIndex's local PyMuPDFReader.
    
    """
    # Initialize the reader once outside the loop to save memory
    reader = PyMuPDFReader()
    extracted_results = {}
    
    for file_path in file_paths:
        # 1. Check if file actually exists before trying to read it
        if not os.path.exists(file_path):
            print(f"⚠️ Warning: File not found, skipping -> {file_path}")
            continue
            
        try:
            
            # 2. Extract the data for this specific file
            documents = reader.load_data(file_path=file_path)
            
            # 3. Combine all pages into one single string
            full_text = "\n".join([doc.text for doc in documents])
            
            # 4. Save to our dictionary
            extracted_results[file_path] = full_text
            
        except Exception as e:
            print(f"❌ Error processing {file_path}: {str(e)}")
            
    print(f"\n✅ Successfully extracted {len(extracted_results)} out of {len(file_paths)} files.")
    return extracted_results

In [17]:
extracted_data = batch_extract_pdfs(candidates_file_paths)

t = []

for path, text in extracted_data.items():
   t.append(text)

candidates_resumes_text = "\n\n".join(t)


✅ Successfully extracted 2 out of 2 files.


## Analyses

In [18]:
query = f"""Based on the following job description, please rank the candidates in order of the most qualified to the least qualified and share the analysis of why specific candidates are suitable for the job.

        Job Description:
        {job_description}

        Candidates:
        {candidates_resumes_text}
        """

analyses = llm.complete(query)

2026-03-05 13:48:36,874 - INFO - HTTP Request: POST https://router.huggingface.co/v1/chat/completions "HTTP/1.1 200 OK"


In [19]:
print(analyses)

Based on the job description and candidate profiles, here is my ranking from most to least qualified:

## 1. SENIOR VICE PRESIDENT OF GLOBAL INFORMATION TECHNOLOGY
**Why most qualified:**
- **8+ years of progressive IT management experience** (1985-2004+ = ~20+ years total)
- **Extensive leadership experience** managing global teams (70+ employees worldwide)
- **Proven track record** of successful large-scale implementations (Oracle ERP, global suites, multiple acquisitions)
- **Strategic alignment** with business objectives - "aligned and supporting key company initiatives"
- **Enterprise systems expertise** - extensive experience with ERP systems, Oracle applications, global implementations
- **Budget management** - consistently managed large budgets ($8M+)
- **Cross-functional collaboration** - worked with M&A teams, global operations
- **Technical depth** - hands-on experience with enterprise software, infrastructure, and cloud solutions
- **Industry experience** - worked in manufa